# Celebal Summer Internship 2026 — Week 2 Task
## E-Commerce Sales Database (ShopEase)

**Approach:** The schema and sample data are loaded into an in-memory **SQLite** database via pandas.
Each question below shows the **question**, the **SQL query**, and the **result table** it produces.

> **Note on RDBMS:** The assignment targets MySQL. SQLite runs all the SELECT/JOIN/GROUP BY queries
> identically. Three questions test engine-specific behaviour (Q6 CHECK, Q23 foreign-key violation,
> Q27 transaction) — these are demonstrated below and annotated with the equivalent MySQL behaviour.

## Setup — create the database, tables, and load data

In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(":memory:")

conn.execute("PRAGMA foreign_keys = ON")   # To enforce Foreign Key constraints which is off by default in SQLite
cur = conn.cursor()
# Helper Function which run a query and return the result as a DataFrame.
def q(sql):
    return pd.read_sql_query(sql, conn)

print("Connected to in-memory SQLite database.")

Connected to in-memory SQLite database.


In [3]:
cur.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    city TEXT NOT NULL,
    state TEXT NOT NULL,
    join_date DATE NOT NULL,
    is_premium INTEGER DEFAULT 0
);
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    brand TEXT NOT NULL,
    unit_price REAL NOT NULL CHECK (unit_price > 0),
    stock_qty INTEGER NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date DATE NOT NULL,
    status TEXT NOT NULL DEFAULT 'Pending'
        CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount REAL NOT NULL CHECK (total_amount >= 0),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    unit_price REAL NOT NULL CHECK (unit_price > 0),
    discount_pct REAL DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
CREATE INDEX idx_customers_city    ON customers(city);
CREATE INDEX idx_customers_state   ON customers(state);
CREATE INDEX idx_products_category ON products(category);
CREATE INDEX idx_orders_date       ON orders(order_date);
CREATE INDEX idx_orders_status     ON orders(status);
""")
conn.commit()
print("Tables and indexes created.")

Tables and indexes created.


In [4]:
cur.executescript("""
INSERT INTO customers VALUES
(101,'Aarav','Sharma','aarav.s@email.com','Mumbai','Maharashtra','2024-01-15',1),
(102,'Priya','Patel','priya.p@email.com','Ahmedabad','Gujarat','2024-02-20',0),
(103,'Rohan','Gupta','rohan.g@email.com','Delhi','Delhi','2024-03-10',1),
(104,'Sneha','Reddy','sneha.r@email.com','Hyderabad','Telangana','2024-04-05',0),
(105,'Vikram','Singh','vikram.s@email.com','Jaipur','Rajasthan','2024-05-12',1),
(106,'Ananya','Iyer','ananya.i@email.com','Chennai','Tamil Nadu','2024-06-18',0),
(107,'Karan','Mehta','karan.m@email.com','Pune','Maharashtra','2024-07-22',1),
(108,'Divya','Nair','divya.n@email.com','Kochi','Kerala','2024-08-30',0);

INSERT INTO products VALUES
(201,'Wireless Earbuds','Electronics','BoAt',1499.00,250),
(202,'Cotton T-Shirt','Clothing','Levis',799.00,500),
(203,'Smart Watch','Electronics','Noise',2999.00,150),
(204,'Running Shoes','Clothing','Nike',4599.00,120),
(205,'Bluetooth Speaker','Electronics','JBL',3499.00,200),
(206,'Bedsheet Set','Home','Spaces',1299.00,300),
(207,'Laptop Stand','Electronics','AmazonBasics',899.00,180),
(208,'Cushion Covers (Set)','Home','HomeCenter',599.00,400);

INSERT INTO orders VALUES
(1001,101,'2024-08-01','Delivered',4498.00),
(1002,102,'2024-08-03','Delivered',799.00),
(1003,103,'2024-08-05','Shipped',7498.00),
(1004,101,'2024-08-10','Delivered',3499.00),
(1005,104,'2024-08-12','Cancelled',2999.00),
(1006,105,'2024-08-15','Delivered',5898.00),
(1007,106,'2024-08-18','Pending',1299.00),
(1008,103,'2024-08-20','Delivered',899.00),
(1009,107,'2024-08-25','Shipped',6098.00),
(1010,108,'2024-08-28','Delivered',1598.00);

INSERT INTO order_items VALUES
(5001,1001,201,2,1499.00,0),(5002,1001,207,1,899.00,10),
(5003,1002,202,1,799.00,0),(5004,1003,203,1,2999.00,0),
(5005,1003,204,1,4599.00,5),(5006,1004,205,1,3499.00,0),
(5007,1005,203,1,2999.00,0),(5008,1006,201,1,1499.00,10),
(5009,1006,204,1,4599.00,5),(5010,1007,206,1,1299.00,0),
(5011,1008,207,1,899.00,0),(5012,1009,205,1,3499.00,0),
(5013,1009,208,2,599.00,15),(5014,1010,206,1,1299.00,0),
(5015,1010,208,1,599.00,0);
""")
conn.commit()
print("Data loaded.")

Data loaded.


---
# Section A — SQL Basics (SELECT, Constraints, Primary Keys)

### Q1. Display all columns and rows from the customers table.

In [5]:
q("SELECT * FROM customers")

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


### Q2. Retrieve only first_name, last_name, and city of all customers.

In [6]:
q("SELECT first_name, last_name, city FROM customers")

,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi


### Q3. List all unique categories available in the products table.

In [7]:
q("SELECT DISTINCT category FROM products")

,category
0,Clothing
1,Electronics
2,Home


### Q4. Identify the Primary Key of each table. Why must a PK be unique and NOT NULL?

**Primary Keys:** `customers.customer_id`, `products.product_id`, `orders.order_id`, `order_items.item_id`.

A Primary Key must be **unique** and **NOT NULL** because it identifies each row in a table.
Uniqueness guarantees no two records share the same identifier; NOT NULL guarantees every record
has a valid ID. Without these rules the database could not reliably identify, update, or relate
records, leading to data inconsistency.

### Q5. Constraints on the email column; what happens on a duplicate email?

Two constraints apply to `email`: **UNIQUE** and **NOT NULL**. Inserting a duplicate email violates
the UNIQUE constraint and the engine rejects it with a *"Duplicate entry"* error (in MySQL), so no
two customers can share an email. NOT NULL also requires every customer to have an email value.

### Q6. Insert a product with `unit_price = -50`. What happens and which constraint prevents it?

The `CHECK (unit_price > 0)` constraint prevents it. Below we attempt the insert and catch the error
so the violation is visible.

In [8]:
# MySQL equivalent:  INSERT INTO products VALUES (209,'Pillow Cover','Home','AmazonBasics',-50,20);
try:
    cur.execute("INSERT INTO products VALUES (209,'Pillow Cover','Home','AmazonBasics',-50,20)")
    conn.commit()
    print("Inserted (unexpected).")
except Exception as e:
    print("Insert rejected — constraint violated:")
    print(" ", e)

# Explanation:
# The CHECK (unit_price > 0) constraint is evaluated before the row is inserted.
# unit_price = -50 fails the condition, so the row is rejected.
# In MySQL this raises: "Check constraint 'products_chk_1' is violated."


Insert rejected — constraint violated:
  CHECK constraint failed: unit_price > 0


---
# Section B — Filtering & Optimization (WHERE, Indexes)

### Q7. Retrieve all orders with status = 'Delivered'.

In [9]:
q("SELECT * FROM orders WHERE status = 'Delivered'")

,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498.0
1,1002,102,2024-08-03,Delivered,799.0
2,1004,101,2024-08-10,Delivered,3499.0
3,1006,105,2024-08-15,Delivered,5898.0
4,1008,103,2024-08-20,Delivered,899.0
5,1010,108,2024-08-28,Delivered,1598.0


### Q8. Products in 'Electronics' with unit_price greater than 2000.

In [10]:
q("SELECT * FROM products WHERE category = 'Electronics' AND unit_price > 2000")

,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999.0,150
1,205,Bluetooth Speaker,Electronics,JBL,3499.0,200


### Q9. Customers who joined in 2024 and belong to state 'Maharashtra'.

In [11]:
q("""
SELECT * FROM customers
WHERE state = 'Maharashtra'
AND join_date >= '2024-01-01' AND join_date < '2025-01-01'
""")

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1


### Q10. Orders between '2024-08-10' and '2024-08-25' (inclusive) that are NOT cancelled.

In [12]:
q("""
SELECT * FROM orders
WHERE status <> 'Cancelled'
AND order_date BETWEEN '2024-08-10' AND '2024-08-25'
""")

,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499.0
1,1006,105,2024-08-15,Delivered,5898.0
2,1007,106,2024-08-18,Pending,1299.0
3,1008,103,2024-08-20,Delivered,899.0
4,1009,107,2024-08-25,Shipped,6098.0


### Q11. What does `idx_orders_date` do? Sample query that benefits from it.

`idx_orders_date` is a B-tree index on `orders(order_date)`. It stores order_date values in sorted
order with pointers to the matching rows, so the engine can jump straight to the relevant dates
instead of scanning every row (a full table scan). This speeds up range and equality filters on
`order_date`, like the query below.

In [13]:
q("""
SELECT * FROM orders
WHERE order_date BETWEEN '2024-08-01' AND '2024-08-15'
""")

,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498.0
1,1002,102,2024-08-03,Delivered,799.0
2,1003,103,2024-08-05,Shipped,7498.0
3,1004,101,2024-08-10,Delivered,3499.0
4,1005,104,2024-08-12,Cancelled,2999.0
5,1006,105,2024-08-15,Delivered,5898.0


### Q12. Would the index on join_date be used for `YEAR(join_date) = 2024`? Rewrite to be SARGable.

**No.** Wrapping the column in a function — `YEAR(join_date)` — makes the predicate non-SARGable:
the engine must compute `YEAR()` for every row before comparing, so it cannot use the sorted index
and falls back to a full table scan. Rewriting it as a range on the raw column lets the index be used:

In [14]:
q("""
SELECT * FROM customers
WHERE join_date >= '2024-01-01' AND join_date < '2025-01-01'
""")

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


---
# Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX)

### Q13. Count the total number of orders.

In [15]:
q("SELECT COUNT(*) AS total_orders FROM orders")

,total_orders
0,10


### Q14. Total revenue from all 'Delivered' orders.

In [16]:
q("""
SELECT SUM(total_amount) AS total_delivered_revenue
FROM orders WHERE status = 'Delivered'
""")

,total_delivered_revenue
0,17191.0


### Q15. Average unit_price of products in each category.

In [17]:
q("""
SELECT category, AVG(unit_price) AS avg_price
FROM products GROUP BY category
""")

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0
2,Home,949.0


### Q16. Per status: count of orders and total revenue, sorted by revenue DESC.

In [18]:
q("""
SELECT status, COUNT(*) AS total_orders, SUM(total_amount) AS total_revenue
FROM orders GROUP BY status ORDER BY total_revenue DESC
""")

,status,total_orders,total_revenue
0,Delivered,6,17191.0
1,Shipped,2,13596.0
2,Cancelled,1,2999.0
3,Pending,1,1299.0


### Q17. Most expensive (MAX) and cheapest (MIN) product in each category.

In [19]:
q("""
SELECT category, MAX(unit_price) AS maximum_price, MIN(unit_price) AS minimum_price
FROM products GROUP BY category
""")

,category,maximum_price,minimum_price
0,Clothing,4599.0,799.0
1,Electronics,3499.0,899.0
2,Home,1299.0,599.0


### Q18. Categories where the average unit_price is greater than 2000 (HAVING).

In [20]:
q("""
SELECT category, AVG(unit_price) AS avg_price
FROM products GROUP BY category HAVING AVG(unit_price) > 2000
""")

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0


---
# Section D — Joins & Relationships

### Q19. INNER JOIN: each order with the customer's first and last name.

In [21]:
q("""
SELECT o.order_id, o.order_date, c.first_name, c.last_name, o.total_amount
FROM customers AS c
INNER JOIN orders AS o ON c.customer_id = o.customer_id
""")

,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498.0
1,1002,2024-08-03,Priya,Patel,799.0
2,1003,2024-08-05,Rohan,Gupta,7498.0
3,1004,2024-08-10,Aarav,Sharma,3499.0
4,1005,2024-08-12,Sneha,Reddy,2999.0
5,1006,2024-08-15,Vikram,Singh,5898.0
6,1007,2024-08-18,Ananya,Iyer,1299.0
7,1008,2024-08-20,Rohan,Gupta,899.0
8,1009,2024-08-25,Karan,Mehta,6098.0
9,1010,2024-08-28,Divya,Nair,1598.0


### Q20. LEFT JOIN: all customers and their orders (NULLs where no order).

In [22]:
q("""
SELECT c.customer_id, c.first_name, c.last_name, o.order_id, o.order_date, o.total_amount
FROM customers AS c
LEFT JOIN orders AS o ON c.customer_id = o.customer_id
""")

,customer_id,first_name,last_name,order_id,order_date,total_amount
0,101,Aarav,Sharma,1001,2024-08-01,4498.0
1,101,Aarav,Sharma,1004,2024-08-10,3499.0
2,102,Priya,Patel,1002,2024-08-03,799.0
3,103,Rohan,Gupta,1003,2024-08-05,7498.0
4,103,Rohan,Gupta,1008,2024-08-20,899.0
5,104,Sneha,Reddy,1005,2024-08-12,2999.0
6,105,Vikram,Singh,1006,2024-08-15,5898.0
7,106,Ananya,Iyer,1007,2024-08-18,1299.0
8,107,Karan,Mehta,1009,2024-08-25,6098.0
9,108,Divya,Nair,1010,2024-08-28,1598.0


### Q21. Three-table JOIN (orders -> order_items -> products).

In [23]:
q("""
SELECT o.order_id, p.product_name, oi.quantity, oi.unit_price, oi.discount_pct
FROM orders AS o
INNER JOIN order_items AS oi ON o.order_id = oi.order_id
INNER JOIN products AS p ON oi.product_id = p.product_id
""")

,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499.0,0.0
1,1001,Laptop Stand,1,899.0,10.0
2,1002,Cotton T-Shirt,1,799.0,0.0
3,1003,Smart Watch,1,2999.0,0.0
4,1003,Running Shoes,1,4599.0,5.0
5,1004,Bluetooth Speaker,1,3499.0,0.0
6,1005,Smart Watch,1,2999.0,0.0
7,1006,Wireless Earbuds,1,1499.0,10.0
8,1006,Running Shoes,1,4599.0,5.0
9,1007,Bedsheet Set,1,1299.0,0.0


### Q22. LEFT JOIN vs RIGHT JOIN; when to use FULL OUTER JOIN.

**LEFT JOIN** returns all rows from the left table plus matching rows from the right (unmatched right
side = NULL). **RIGHT JOIN** does the reverse.
- `customers LEFT JOIN orders` → every customer, even those with no orders.
- `customers RIGHT JOIN orders` → every order, even if its customer were missing.

A **FULL OUTER JOIN** returns all rows from both tables, matched where possible and NULL where not —
useful to find unmatched rows on either side at once. MySQL has no FULL OUTER JOIN keyword; it is
emulated with a `LEFT JOIN` `UNION` a `RIGHT JOIN`.

### Q23. Foreign Key relationships; inserting an order with customer_id = 999.

**Foreign Keys:** `orders.customer_id → customers.customer_id`, `order_items.order_id → orders.order_id`,
`order_items.product_id → products.product_id`.

Inserting an order with `customer_id = 999` (which doesn't exist) violates the FK constraint. The
demonstration below catches the resulting error.

In [24]:
try:
    cur.execute("INSERT INTO orders VALUES (9999, 999, '2024-09-01', 'Pending', 100.00)")
    conn.commit()
    print("Inserted (unexpected).")
except Exception as e:
    print("Insert rejected — foreign key constraint failed:")
    print(" ", e)

# In MySQL this raises:
# "Cannot add or update a child row: a foreign key constraint fails"
# This enforces referential integrity — every order must point to a real customer.


Insert rejected — foreign key constraint failed:
  FOREIGN KEY constraint failed


---
# Section E — Advanced Concepts (CASE, ACID, Transactions)

### Q24. CASE to classify products into price tiers.

In [25]:
q("""
SELECT product_name, unit_price,
    CASE
        WHEN unit_price < 1000 THEN 'Budget'
        WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
        ELSE 'Premium'
    END AS price_tier
FROM products
""")

,product_name,unit_price,price_tier
0,Wireless Earbuds,1499.0,Mid-Range
1,Cotton T-Shirt,799.0,Budget
2,Smart Watch,2999.0,Mid-Range
3,Running Shoes,4599.0,Premium
4,Bluetooth Speaker,3499.0,Premium
5,Bedsheet Set,1299.0,Mid-Range
6,Laptop Stand,899.0,Budget
7,Cushion Covers (Set),599.0,Budget


### Q25. CASE inside an aggregate: 'Delivered' vs 'Not Delivered' in a single row.

In [26]:
q("""
SELECT
    SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN status <> 'Delivered' THEN 1 ELSE 0 END) AS not_delivered_orders
FROM orders
""")

,delivered_orders,not_delivered_orders
0,6,4


### Q26. Explain ACID with a bank-transfer example.

Using a bank transfer (₹1000 from account A to account B):

- **A – Atomicity:** All steps succeed or none do. The debit from A and credit to B happen together;
  if the credit fails, the debit is rolled back, so money never disappears.
- **C – Consistency:** A transaction moves the database from one valid state to another, honouring
  all constraints. Total money across both accounts is the same before and after.
- **I – Isolation:** Concurrent transactions don't interfere. Two simultaneous transfers each see a
  consistent view and don't observe each other's half-finished work.
- **D – Durability:** Once committed, the result survives crashes or power loss. After the transfer
  commits, the new balances persist even if the server restarts immediately.

### Q27. Atomic transaction — insert order, insert two items, update stock; COMMIT/ROLLBACK.

The MySQL transaction block is shown first, then demonstrated in SQLite.
(Order total 2098 = item 201 at 1499 + item 208 at 599, so the order reconciles with its line items.)

```sql
START TRANSACTION;

INSERT INTO orders (order_id, customer_id, order_date, status, total_amount)
VALUES (1011, 102, CURDATE(), 'Pending', 2098.00);

INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price, discount_pct)
VALUES (5016, 1011, 201, 1, 1499.00, 0),
       (5017, 1011, 208, 1,  599.00, 0);

UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 201;
UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 208;

COMMIT;   -- run ROLLBACK; instead if any statement above fails
```

In [27]:
# Demonstration in SQLite (manual transaction control: BEGIN ... COMMIT / ROLLBACK).
# isolation_level=None lets us issue BEGIN/COMMIT/ROLLBACK ourselves, mirroring MySQL.
conn.isolation_level = None
try:
    cur.execute("BEGIN")
    cur.execute("""INSERT INTO orders (order_id, customer_id, order_date, status, total_amount)
                   VALUES (1011, 102, date('now'), 'Pending', 2098.00)""")
    cur.executemany("""INSERT INTO order_items
                       (item_id, order_id, product_id, quantity, unit_price, discount_pct)
                       VALUES (?,?,?,?,?,?)""",
                    [(5016,1011,201,1,1499.00,0),(5017,1011,208,1,599.00,0)])
    cur.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 201")
    cur.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 208")
    cur.execute("COMMIT")
    print("Transaction committed successfully.")
except Exception as e:
    cur.execute("ROLLBACK")
    print("Transaction rolled back due to error:", e)

# Verify the new order and updated stock
display(q("SELECT * FROM orders WHERE order_id = 1011"))
display(q("SELECT product_id, product_name, stock_qty FROM products WHERE product_id IN (201,208)"))


Transaction committed successfully.


,order_id,customer_id,order_date,status,total_amount
0,1011,102,2026-06-01,Pending,2098.0


,product_id,product_name,stock_qty
0,201,Wireless Earbuds,249
1,208,Cushion Covers (Set),399


---
# Insights & Summary

A short business read of the ShopEase sample data, drawn from the queries above:

1. **Revenue concentrates in completed orders.** Delivered orders are the bulk of realised revenue
   (Q14/Q16); Pending and Cancelled orders represent revenue that is at risk or already lost. Reducing
   cancellations would directly lift the realised total.

2. **Electronics is the premium, high-value category.** It holds the widest price range — from the
   Laptop Stand at the low end up to the Bluetooth Speaker — and is the category whose average price
   clears ₹2000 (Q15/Q17/Q18). Clothing carries the single most expensive item (Running Shoes), while
   Home is consistently the budget category.

3. **Order status is the key health metric.** Grouping by status (Q16, Q25) shows most orders reach
   Delivered, but a non-trivial share sit in Shipped/Pending/Cancelled. Tracking the Delivered vs
   Not-Delivered split over time is a simple fulfilment-health KPI.

4. **The schema is sound and enforces integrity.** Primary keys, the CHECK constraints (Q6), and the
   foreign keys (Q23) actively reject bad data — negative prices and orphan orders are blocked at the
   database level, not just in application code.

5. **Indexing matters for the common filters.** Date-range and category/status filters (the queries in
   Section B) are exactly the access patterns the provided indexes target. Keeping predicates SARGable
   (Q12) ensures those indexes are actually used as the data grows.